### Синтаксические метрики для корпуса книг (syntaxcomp)

Эта тетрадка:

- берёт тексты книг из `books_clean/` по `corpus_metadata.csv`
- размечает их **UDPipe** (CoNLL-U) через публичный API UFAL
- считает синтаксические метрики библиотекой [`syntaxcomp`](https://pypi.org/project/syntaxcomp/)
- сохраняет кэш разметки `.conllu` и итоговую таблицу `.csv`

Важно:

- `syntaxcomp` гарантирует корректность в первую очередь для разметки **UDPipe 2.12**.
- Есть предупреждение о баге в извлечении **T-unit**. Поэтому метрики, завязанные на T-unit (`num_tu`, `mtl`, `cpt`), внизу можно автоматически исключить из финального датасета.


In [112]:
from __future__ import annotations

from pathlib import Path

# Корень репозитория (тетрадка лежит в python_syntax part/)
ROOT = Path.cwd().resolve().parent

DATA_DIR = ROOT / "books_dontsova_full_clean"
# DATA_DIR = ROOT / "books_clean"
META_CSV = ROOT / "corpus_metadata_dontsova_full.csv"
# META_CSV = ROOT / "corpus_metadata.csv"

CACHE_DIR = ROOT / "python_syntax part" / "dontsova_full_cache_conllu_udpipe"
# CACHE_DIR = ROOT / "python_syntax part" / "cache_conllu_udpipe"
OUT_DIR = ROOT / "python_syntax part" / "dontsova_full_output"
# OUT_DIR = ROOT / "python_syntax part" / "output"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROOT, DATA_DIR, META_CSV, CACHE_DIR, OUT_DIR

(WindowsPath('C:/Users/Маша/maga/andan_project_26'),
 WindowsPath('C:/Users/Маша/maga/andan_project_26/books_dontsova_full_clean'),
 WindowsPath('C:/Users/Маша/maga/andan_project_26/corpus_metadata_dontsova_full.csv'),
 WindowsPath('C:/Users/Маша/maga/andan_project_26/python_syntax part/dontsova_full_cache_conllu_udpipe'),
 WindowsPath('C:/Users/Маша/maga/andan_project_26/python_syntax part/dontsova_full_output'))

In [96]:
# Если запускаете впервые, раскомментируйте установку.
# (В идеале — в отдельном venv.)

# %pip install -q syntaxcomp conllu textdistance pandas requests tqdm

In [109]:
import json
import time
from dataclasses import dataclass

import pandas as pd
import requests
from tqdm.auto import tqdm

from syntaxcomp.complexity import TextComplexity

In [110]:
import hashlib
import random
import re
from typing import Iterable

UDPIPE_API = "https://lindat.mff.cuni.cz/services/udpipe/api"
UDPIPE_MODELS_URL = f"{UDPIPE_API}/models"
UDPIPE_PROCESS_URL = f"{UDPIPE_API}/process"

PREFER_UD_VERSION = "ud-2.12"  # syntaxcomp советует UDPipe 2.12

# Практика показывает, что для длинных художественных текстов лучше меньше чанки,
# иначе сервис иногда закрывает соединение без ответа.
MAX_CHUNK_CHARS = 10_000
REQUEST_TIMEOUT_S = 180

# Пауза между обычными запросами
SLEEP_BETWEEN_REQUESTS_S = 0.4

# Ретраи при сетевых сбоях/перегрузке сервиса
UDPIPE_MAX_RETRIES = 6
UDPIPE_BACKOFF_BASE_S = 1.0
UDPIPE_BACKOFF_MAX_S = 40.0
UDPIPE_BACKOFF_JITTER_S = 0.25

FORCE_REPARSE = False  # True => пересоздать весь кэш даже если файлы есть

DROP_TUNIT_METRICS = True  # из-за известного бага T-unit extraction в syntaxcomp

print("UDPipe:", UDPIPE_PROCESS_URL)

UDPipe: https://lindat.mff.cuni.cz/services/udpipe/api/process


In [113]:
def load_metadata() -> pd.DataFrame:
    df = pd.read_csv(META_CSV, encoding="utf-8")
    # Чистим возможный BOM (у тебя в скриптах было utf-8-sig)
    if df.columns[0].startswith("\ufeff"):
        df = df.rename(columns={df.columns[0]: df.columns[0].replace("\ufeff", "")})
    return df


def book_txt_path(row: pd.Series) -> Path:
    # Берём чистые книги (не токены/леммы), чтобы UDPipe видел пунктуацию.
    return DATA_DIR / row["series_slug"] / row["имя_файла"]


def cache_conllu_path(row: pd.Series, model_id: str) -> Path:
    # Чтобы кэш был воспроизводимым при смене модели, включаем model_id в имя.
    safe_model = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_id)
    return CACHE_DIR / f"{row['series_slug']}__{row['имя_файла']}.{safe_model}.conllu"


def cache_meta_path(conllu_path: Path) -> Path:
    return conllu_path.with_suffix(conllu_path.suffix + ".json")


meta_df = load_metadata()
# meta_df.head(100)
len(meta_df)

288

In [71]:
r = requests.get(UDPIPE_MODELS_URL, timeout=REQUEST_TIMEOUT_S)
r.raise_for_status()
payload = r.json()
models = payload.get("models", {})

lang_lower = LANGUAGE.lower()
candidates = [m for m in models.keys() if m.lower().startswith('russian') and PREFER_UD_VERSION in m.lower()]

In [72]:
print('\n'.join(candidates))

russian-syntagrus-ud-2.12-230717
russian-gsd-ud-2.12-230717
russian-taiga-ud-2.12-230717


In [101]:
UDPIPE_MODEL = "russian-syntagrus-ud-2.12-230717" # обучен на художественных текстах, поэтому выбираем его

In [100]:
def chunk_text(text: str, max_chars: int = MAX_CHUNK_CHARS) -> list[str]:
    """Режет текст на куски, чтобы UDPipe принимал запросы. Режет по пустым строкам/абзацам.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]

    chunks: list[str] = []
    buf = ""

    # добавляет чанк с саиску и очищает buf
    def flush():
        nonlocal buf
        if buf.strip():
            chunks.append(buf.strip())
        buf = ""

    for block in blocks:
        if len(block) > max_chars:
            flush()
            lines = [ln.strip() for ln in block.split("\n") if ln.strip()]
            part = ""
            for ln in lines:
                if len(part) + 1 + len(ln) > max_chars:
                    if part.strip():
                        chunks.append(part.strip())
                    part = ln
                else:
                    part = (part + "\n" + ln).strip() if part else ln
            if part.strip():
                chunks.append(part.strip())
            continue

        if not buf:
            buf = block
        elif len(buf) + 2 + len(block) <= max_chars:
            buf = buf + "\n\n" + block
        else:
            flush()
            buf = block

    flush()
    return chunks


In [102]:
_session = requests.Session()


def _sleep_backoff(attempt: int) -> None:
    backoff = min(UDPIPE_BACKOFF_MAX_S, UDPIPE_BACKOFF_BASE_S * (2**attempt))
    backoff += random.random() * UDPIPE_BACKOFF_JITTER_S
    time.sleep(backoff)


def udpipe_process(text: str, model: str = UDPIPE_MODEL) -> str:
    """Возвращает CoNLL-U разметку для plain text.

    UDPipe иногда рвёт соединение без ответа (RemoteDisconnected). Поэтому тут есть ретраи.
    """
    data = {
        "model": model,
        "tokenizer": "",
        "tagger": "",
        "parser": "",
        "data": text,
        "output": "conllu",
    }

    last_err: Exception | None = None
    for attempt in range(UDPIPE_MAX_RETRIES + 1):
        try:
            r = _session.post(UDPIPE_PROCESS_URL, data=data, timeout=REQUEST_TIMEOUT_S)

            # 429/5xx — часто временная перегрузка; попробуем повторить
            if r.status_code in {429, 502, 503, 504}:
                last_err = requests.HTTPError(f"HTTP {r.status_code}: {r.text[:200]}")
                if attempt < UDPIPE_MAX_RETRIES:
                    _sleep_backoff(attempt)
                    continue
                raise last_err

            r.raise_for_status()
            payload = r.json()
            if "result" not in payload:
                raise RuntimeError(
                    f"UDPipe response has no 'result': {list(payload.keys())}"
                )
            return payload["result"]

        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout, ValueError) as e:
            last_err = e
            if attempt < UDPIPE_MAX_RETRIES:
                _sleep_backoff(attempt)
                continue
            raise

    assert last_err is not None
    raise last_err


def annotate_text_to_conllu(text: str, model: str = UDPIPE_MODEL) -> str:
    chunks = chunk_text(text)
    conllus: list[str] = []
    for ch in tqdm(chunks, desc="UDPipe chunks", leave=False):
        conllus.append(udpipe_process(ch, model=model).strip())
        time.sleep(SLEEP_BETWEEN_REQUESTS_S)
    return "\n\n".join([c for c in conllus if c]) + "\n"

In [103]:
def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def annotate_book(row: pd.Series, model: str = UDPIPE_MODEL, force: bool = FORCE_REPARSE) -> Path:
    src = book_txt_path(row)
    print(src)
    text = src.read_text(encoding="utf-8")
    text_hash = sha256_text(text)

    conllu_path = cache_conllu_path(row, model_id=model)
    meta_path = cache_meta_path(conllu_path)

    if not force and conllu_path.exists() and meta_path.exists():
        try:
            old = json.loads(meta_path.read_text(encoding="utf-8"))
            if old.get("text_sha256") == text_hash and old.get("model") == model:
                return conllu_path
        except Exception:
            pass

    conllu = annotate_text_to_conllu(text, model=model)
    conllu_path.write_text(conllu, encoding="utf-8")
    meta_path.write_text(
        json.dumps(
            {
                "src": str(src.relative_to(ROOT)),
                "model": model,
                "text_sha256": text_hash,
                "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return conllu_path

In [105]:
from time import perf_counter
from conllu import parse

# Для целой книги lev_pos/lev_dep в syntaxcomp считаются как попарные расстояния
# между ВСЕМИ предложениями => O(n^2). Поэтому считаем Levenshtein на подвыборке.
LEV_SAMPLE_SENTENCES = 250  # 200-400 обычно достаточно; 0 => выключить lev_* (будут 0)


def compute_syntaxcomp_metrics_from_conllu(conllu_text: str) -> dict:
    sentences = parse(conllu_text)
    print(f"Sentences in conllu: {len(sentences)}")

    # Временно подменяем pairwise_levenshtein, чтобы не считать на всех парах.
    orig_pairwise = getattr(TextComplexity, "pairwise_levenshtein", None)

    def _pairwise_levenshtein_sample(chains):
        import random
        from itertools import combinations
        from textdistance import levenshtein

        if LEV_SAMPLE_SENTENCES and len(chains) > LEV_SAMPLE_SENTENCES:
            chains = random.sample(chains, LEV_SAMPLE_SENTENCES)
        elif not LEV_SAMPLE_SENTENCES:
            return []

        return [levenshtein.distance(a, b) for a, b in combinations(chains, 2)]

    if orig_pairwise is not None:
        TextComplexity.pairwise_levenshtein = staticmethod(_pairwise_levenshtein_sample)

    t0 = perf_counter()
    try:
        tc = TextComplexity(sentences)
        # В некоторых версиях syntaxcomp tc.info() только печатает и возвращает None.
        # Поэтому собираем метрики из атрибутов tc.
        metrics = {
            "Number of Sentences": getattr(tc, "num_s", None),
            "Number of Words": getattr(tc, "num_w", None),
            "Number of Clauses": getattr(tc, "num_cl", None),
            # "Number of T-Units": getattr(tc, "num_tu", None),
            "Mean Sentence Length": round(getattr(tc, "msl", 0.0), 2),
            "Mean Clause Length": round(getattr(tc, "mcl", 0.0), 2),
            # "Mean T-Unit Length": round(getattr(tc, "mtl", 0.0), 2),
            "Mean Number of Clauses per Sentence": round(getattr(tc, "cps", 0.0), 2),
            # "Mean Number of Clauses per T-Unit": round(getattr(tc, "cpt", 0.0), 2),
            "Mean Tree Depth": round(getattr(tc, "mtd", 0.0), 2),
            "Median Tree Depth": getattr(tc, "mdtd", None),
            "Minimum Tree Depth": getattr(tc, "mntd", None),
            "Maximum Tree Depth": getattr(tc, "mxtd", None),
            "Mean Dependency Distance": round(getattr(tc, "mdd", 0.0), 2),
            "Node-to-Terminal-Node Ratio": round(getattr(tc, "node_to_term", 0.0), 2),
            "Average Levenshtein Distance between POS": round(getattr(tc, "lev_pos", 0.0), 2),
            "Average Levenshtein Distance between deprel": round(getattr(tc, "lev_dep", 0.0), 2),
            "Average NP Length": round(getattr(tc, "avg_np_len", 0.0), 2),
            "Complex NP Ratio": round(getattr(tc, "comp_np_ratio", 0.0), 2),
            "Number of Combined Clauses": getattr(tc, "comb", None),
            "Number of Coordinate Clauses": getattr(tc, "coord", None),
            "Number of Subordinate Clauses": getattr(tc, "subord", None),
            "Coordinate to Combined Clause Ratio": round(getattr(tc, "coord_to_comb", 0.0), 2),
            "Subordinate to Combined Clause Ratio": round(getattr(tc, "subord_to_comb", 0.0), 2),
            "Coordinate to Subordinate Clause Ratio": round(getattr(tc, "coord_to_subord", 0.0), 2),
            "Coordinate Clause to Sentence Ratio": round(getattr(tc, "coord_to_sent", 0.0), 2),
            "Subordinate Clause to Sentence Ratio": round(getattr(tc, "subord_to_sent", 0.0), 2),
        }

        clause_pct = getattr(tc, "clause_percentage", None)
        if isinstance(clause_pct, dict):
            for rel, val in clause_pct.items():
                metrics[f"{rel}_ratio"] = val

    finally:
        if orig_pairwise is not None:
            TextComplexity.pairwise_levenshtein = orig_pairwise

    metrics["__compute_seconds__"] = round(perf_counter() - t0, 2)

    return metrics


# Smoke-test на одной книге (можно менять индекс)
row0 = meta_df.iloc[1]
conllu_path0 = annotate_book(row0)
conllu0 = conllu_path0.read_text(encoding="utf-8")
metrics0 = compute_syntaxcomp_metrics_from_conllu(conllu0)
list(metrics0.items())[:12]

C:\Users\Маша\maga\andan_project_26\books_dontsova_full_clean\viola\ВТ_Балерина в бахилах.txt


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Маша\\maga\\andan_project_26\\books_dontsova_full_clean\\viola\\ВТ_Балерина в бахилах.txt'

In [42]:
# conllu0

In [86]:
def compute_metrics_for_corpus(df: pd.DataFrame, model: str = UDPIPE_MODEL) -> pd.DataFrame:
    rows = []
    errors = []

    for _, r in tqdm(df.iterrows(), total=len(df), desc="Books"):
        try:
            conllu_path = annotate_book(r, model=model)
            conllu_text = conllu_path.read_text(encoding="utf-8")
            m = compute_syntaxcomp_metrics_from_conllu(conllu_text)

            out = {
                "автор": r["автор"],
                "название": r["название"],
                "серия": r["серия"],
                "год": int(r["год"]),
                "папка": r["series_slug"],
                "имя_файла": r["имя_файла"],
                "conllu_cache": str(conllu_path.relative_to(ROOT)),
                **m,
            }
            rows.append(out)
        except Exception as e:
            errors.append({"имя_файла": r.get("имя_файла"), "error": repr(e)})

    if errors:
        (OUT_DIR / "syntaxcomp_errors.json").write_text(
            json.dumps(errors, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print(f"Errors: {len(errors)} (см. {OUT_DIR / 'syntaxcomp_errors.json'})")

    return pd.DataFrame(rows)


# Запуск на всём корпусе (может занять время, особенно в первый раз)
df_metrics = compute_metrics_for_corpus(meta_df)
out_csv = OUT_DIR / "syntaxcomp_metrics.csv"
df_metrics.to_csv(out_csv, index=False, encoding="utf-8")
out_csv

Books:   0%|          | 0/298 [00:00<?, ?it/s]

Errors: 298 (см. C:\Users\Маша\maga\andan_project_26\python_syntax part\dontsova_full_output\syntaxcomp_errors.json)


WindowsPath('C:/Users/Маша/maga/andan_project_26/python_syntax part/dontsova_full_output/syntaxcomp_metrics.csv')